# Exhaust Manifolds

These are 3D printable exhaust manifolds which connects a muffler tube to exhaust tips. There are 2 manifolds, one driver and one passenger. The manifolds are further subdivided into right and left sides so that inner supports can be accessed after 3D printing. Each side includes a guide so that the subassemblies can be aligned.

The outer diameter of the tubing is 2.5", and the inlets are connected to midpipes with slide on clamps. The exhaust tips have their own clamps which fit over the tubing. 

## Creating the manifolds

The data looks correct, so now we need to build the manifold in cadquery. To support splitting the design into multiple 3D printing meshes, the function allows variable sweep and trim profiles. 

In [ ]:
%load_ext autoreload
%autoreload 2

from build import Builder, Logger, AppConfig
from config import Configurator
from build123d import *
import cadquery as cq
from jupyter_cadquery import *
import logging

logging.basicConfig(filename="out.txt", level=logging.DEBUG, filemode="w")

model = AppConfig()
logger = Logger()
builder = Builder(logger=logger, config=model)
config = Configurator(logger=logger, builder=builder, config=model)

driver_wire, passenger_wire = builder.create_wire("driver"), builder.create_wire("passenger")
show(driver_wire, passenger_wire)

Examine different tube profiles.

In [ ]:
a, b = builder.create_profile(315, 90, joint_space=0.3), builder.create_profile(45, 90, joint_space=0.3)
c, d = builder.create_profile(180, 30, joint_space=0.3), builder.create_profile(210, 30, joint_space=0.3)
show(a, b, c, d)

profile = builder.create_profile_sketch(180, lap_joint=True)
show(profile)

profile = builder.create_profile_sketch(360)
show(profile)

Build the manifolds so we can perform some basic verification of the shapes.

In [ ]:
bounds = builder.config.bound_box
driver_manifold, passenger_manifold = builder.build_tube("driver"), builder.build_tube("passenger")
assy = cq.Assembly()
assy.add(cq.Shape.cast(driver_manifold.wrapped), name="driver_manifold", color="red")
assy.add(cq.Shape.cast(passenger_manifold.wrapped), name="passenger_manifold", color="green")
assy.add(cq.Shape.cast(bounds.wrapped), name="bounds", color=cq.Color(0.0, 1.0, 1.0, 0.2))
show(assy)

Build individual parts.

In [ ]:
bounds = builder.config.bound_box
assy = cq.Assembly()
assy.add(cq.Shape.cast(builder.build_part("driver").wrapped), name="driver_left", color="red")
assy.add(cq.Shape.cast(builder.build_part("driver", right=True).wrapped), name="driver_right", color="green")
assy.add(cq.Shape.cast(builder.build_part("passenger").wrapped), name="passenger_left", color="purple")
assy.add(cq.Shape.cast(builder.build_part("passenger", right=True).wrapped), name="passenger_right", color="yellow")
assy.add(cq.Shape.cast(bounds.wrapped), name="bounds", color=cq.Color(0.0, 1.0, 1.0, 0.2))
show(assy)

Overlay part positions with parts to see if we are estimating them correctly. This takes a minute to generate.

In [ ]:
driver_left, driver_right = builder.build_part("driver"), builder.build_part("driver", right=True)
passenger_left, passenger_right = builder.build_part("passenger"), builder.build_part("passenger", right=True)


def build_part_position_point(name, offset, right=False):
    """Build a part position point for the part position at offset."""
    tube = builder.build_part(name, right=right, tube_only=True)
    path = builder.create_wire(name)
    center = config.get_part_position(tube, path, offset)
    point = Pos(center) * Sphere(radius=10)
    return point


def build_solid_center_point(name, right=False):
    """Build a solid center point for the part."""
    tube = builder.build_part(name, right=right, tube_only=True)
    center = tube.center()
    point = Pos(center) * Cone(bottom_radius=10, top_radius=0, height=10, align=(Align.CENTER, Align.CENTER, Align.MIN))
    return point


assy = cq.Assembly()
assy.add(cq.Shape.cast(driver_left.wrapped), name="driver_left", color=cq.Color(1.0, 0.0, 0.0, 0.2))
assy.add(cq.Shape.cast(driver_right.wrapped), name="driver_right", color=cq.Color(0.0, 1.0, 0.0, 0.2))
assy.add(cq.Shape.cast(passenger_left.wrapped), name="passenger_left", color=cq.Color(1.0, 0.0, 1.0, 0.2))
assy.add(cq.Shape.cast(passenger_right.wrapped), name="passenger_right", color=cq.Color(1.0, 1.0, 0.0, 0.2))

# Add part positions to the output
[assy.add(cq.Shape.cast(build_part_position_point("driver", offset / 10).wrapped), color="red") for offset in range(10)]
[
    assy.add(cq.Shape.cast(build_part_position_point("driver", offset / 10, right=True).wrapped), color="green")
    for offset in range(10)
]
[
    assy.add(cq.Shape.cast(build_part_position_point("passenger", offset / 10).wrapped), color="purple")
    for offset in range(10)
]
[
    assy.add(cq.Shape.cast(build_part_position_point("passenger", offset / 10, right=True).wrapped), color="yellow")
    for offset in range(10)
]

# Add solid centers to the output
assy.add(cq.Shape.cast(build_solid_center_point("driver").wrapped), color="red")
assy.add(cq.Shape.cast(build_solid_center_point("driver", right=True).wrapped), color="green")
assy.add(cq.Shape.cast(build_solid_center_point("passenger").wrapped), color="purple")
assy.add(cq.Shape.cast(build_solid_center_point("passenger", right=True).wrapped), color="yellow")
show(assy)

As a sanity check, show that we can recombine the parts back into the original manifold shape.

In [ ]:
bounds = builder.config.bound_box
driver_manifold, passenger_manifold = builder.build_tube("driver"), builder.build_tube("passenger")
driver_left, driver_right = builder.build_part("driver"), builder.build_part("driver", right=True)
passenger_left, passenger_right = builder.build_part("passenger"), builder.build_part("passenger", right=True)

assy = cq.Assembly()
assy.add(cq.Shape.cast(driver_manifold.wrapped), name="driver_manifold", color="red")
assy.add(cq.Shape.cast((driver_left + driver_right).wrapped), name="driver_manifold_from_parts", color="blue")
assy.add(cq.Shape.cast(passenger_manifold.wrapped), name="passenger_manifold", color="green")
assy.add(
    cq.Shape.cast((passenger_left + passenger_right).wrapped), name="passenger_manifold_from_parts", color="yellow"
)
assy.add(cq.Shape.cast(bounds.wrapped), name="bounds", color=cq.Color(0.0, 1.0, 1.0, 0.2))
show(assy)